# 04 · Optimization — the engines under the models

Both regressions in `dolcestat` are fit by the same family of **optimizers**:
algorithms that start from some weights and iteratively reduce a loss. You met
them through the model wrappers in notebooks 02–03; here we drive them directly
to understand how they behave — and *when the choice of optimizer matters most*.

The short answer: it matters on **ill-conditioned** problems, where the loss
surface is a long, narrow valley. That's exactly where momentum and Newton's
method earn their keep, so we'll build such a problem on purpose.

## An ill-conditioned problem

The difficulty of an optimization problem is captured by the **condition
number** κ of the loss curvature (the Hessian, which for least squares is
proportional to XᵀX). When κ is large the loss is a stretched valley: steep
across the narrow direction, nearly flat along the long one.

An easy way to create this is to leave features on **very different scales** —
here one feature is ~12× larger than the other. (This is precisely the situation
the scaling in [`01_preprocessing`](01_preprocessing.ipynb) exists to prevent,
as we'll confirm at the end.)

In [ ]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet
from dolcestat.optimization.gradient_descent import GradientDescent
from dolcestat.optimization.newton import NewtonMethod
from dolcestat.optimization.analyzer import OptimizerAnalyzer

rng = np.random.default_rng(0)
n = 200
x1 = rng.normal(0, 1, n)          # feature on a small scale
x2 = rng.normal(0, 1, n) * 12     # feature on a ~12x larger scale
y = 1.5 * x1 + 0.1 * x2 + 0.5 + rng.normal(0, 0.3, n)

df = pl.DataFrame({"x1": x1, "x2": x2, "y": y})
data = DolceSet()
data.load_from_polars_dataframe(df, target_col="y")

X_bias = np.column_stack((data.X, np.ones(n)))
kappa = np.linalg.cond(X_bias.T @ X_bias)
print(f"condition number of XᵀX ≈ {kappa:,.0f}   (large ⇒ ill-conditioned)")

## Plain gradient descent

Gradient descent can only take a step up to a maximum stable size, and that
ceiling *shrinks* as κ grows — so on an ill-conditioned problem we're forced into
a small learning rate. Worse, the number of steps to converge grows roughly
**linearly in κ**: plain GD zig-zags slowly down the valley.

In [ ]:
plain = GradientDescent(data=data, loss_function="mse", flavor="batch",
                        alpha=0.003, n_iters=20000, tol=1e-8)
plain.fit()

print("plain gradient descent:", len(plain.get_loss()), "iterations")
OptimizerAnalyzer(plain).plot_loss()

## Momentum: Polyak and Nesterov

Momentum carries a fraction of the previous step into the next, damping the
zig-zag and letting the optimizer build speed along the valley floor.
`dolcestat` offers two variants via `momentum_type`:

- **`"polyak"`** (heavy ball) — add a fraction of the last step.
- **`"nesterov"`** — look ahead to where momentum is carrying you, then correct.

Both cut the iteration count from ~linear in κ toward ~**√κ** — a big win here.
(The trade-off: momentum tolerates a smaller maximum step, so at very aggressive
learning rates it can overshoot.)

In [ ]:
polyak = GradientDescent(data=data, loss_function="mse", flavor="batch",
                         alpha=0.003, n_iters=20000, tol=1e-8,
                         momentum_type="polyak", momentum_rate=0.9)
nesterov = GradientDescent(data=data, loss_function="mse", flavor="batch",
                           alpha=0.003, n_iters=20000, tol=1e-8,
                           momentum_type="nesterov", momentum_rate=0.9)
polyak.fit()
nesterov.fit()

print("plain   :", len(plain.get_loss()), "iterations")
print("polyak  :", len(polyak.get_loss()), "iterations")
print("nesterov:", len(nesterov.get_loss()), "iterations")

OptimizerAnalyzer([plain, polyak, nesterov],
                  labels=["plain", "polyak", "nesterov"]).plot_loss(log_scale=True)

## Newton's method

Gradient descent uses only the slope. **Newton's method** also uses the
curvature — it inverts the Hessian to rescale the step — which makes it
essentially **independent of κ**: it reaches the minimum of a quadratic loss in
a handful of iterations no matter how stretched the valley. Nesterov momentum
already cut the iteration count to ~√κ; Newton goes further still, since it
doesn't scale with κ at all. That power costs a matrix inversion per step, and
it's the default optimizer for logistic regression.

In [ ]:
newton = NewtonMethod(data=data, loss_function="mse", n_iters=100, tol=1e-8)
newton.fit()

print("nesterov:", len(nesterov.get_loss()), "iterations   |   Newton:", len(newton.get_loss()), "iterations")
OptimizerAnalyzer([nesterov, newton], labels=["nesterov", "newton"]).plot_loss(log_scale=True)

## The cheap fix: scaling

You usually don't need heavy machinery to beat ill-conditioning — you can just
remove it. **Standardizing** the features (notebook 01) puts them on a common
scale, which collapses κ back toward 1. Watch plain gradient descent go from
thousands of iterations to a handful:

In [ ]:
scaled = DolceSet()
scaled.load_from_polars_dataframe(df, target_col="y")
scaled.scale("standardize")          # put both features on a common scale

Xb = np.column_stack((scaled.X, np.ones(n)))
print(f"condition number after scaling ≈ {np.linalg.cond(Xb.T @ Xb):,.1f}")

well_conditioned = GradientDescent(data=scaled, loss_function="mse", flavor="batch",
                                   alpha=0.4, n_iters=20000, tol=1e-8)
well_conditioned.fit()
print("plain gradient descent on scaled data:", len(well_conditioned.get_loss()), "iterations")

## A different axis: how much data per step

Conditioning aside, the `flavor` controls how many rows each step uses — a
trade-off between step cost and step noise (shown here on the well-conditioned,
scaled data):

- **`"batch"`** — the whole dataset every step: smooth but heavier.
- **`"mini_batch"`** — a random `batch_fraction` of rows: a middle ground.
- **`"sgd"`** — a single random row: cheap, noisy steps.

Passing several optimizers to one `OptimizerAnalyzer` overlays their curves.

In [ ]:
batch = GradientDescent(data=scaled, loss_function="mse", flavor="batch",
                        alpha=0.3, n_iters=500)
mini  = GradientDescent(data=scaled, loss_function="mse", flavor="mini_batch",
                        batch_fraction=0.2, alpha=0.3, n_iters=500)
sgd   = GradientDescent(data=scaled, loss_function="mse", flavor="sgd",
                        alpha=0.1, n_iters=500)
for opt in (batch, mini, sgd):
    opt.fit()

OptimizerAnalyzer([batch, mini, sgd]).plot_loss()

## A different problem: overfitting, not conditioning

Everything so far tuned *how fast* we reach the least-squares solution. But
sometimes that solution is the problem. With many correlated features and few
rows, the unconstrained fit has enough freedom to chase noise: it drives the
training loss almost to zero while generalizing badly, propping up large
coefficients that cancel each other out.

**Regularization** attacks that directly by adding a penalty on coefficient size
to the objective, trading a little training fit for a lot of stability. Every
`GradientDescent` accepts two coefficients:

- **`l1`** (Lasso) — adds `l1 * |w|`, pushing weights toward zero individually.
- **`l2`** (Ridge) — adds `(l2 / 2) * ||w||²`, shrinking all weights smoothly.
- Both non-zero together gives **ElasticNet**.

Two details worth knowing. The `l2` gradient term is `l2 * w`, so the penalty is
*half* the squared norm — to match `sklearn.linear_model.Ridge(alpha=a)` on n
rows, pass `l2 = 2 * a / n`. And the bias is never penalized: it only absorbs the
mean of y, so shrinking it would just drag every prediction toward zero.

Let's build a problem that needs this — 25 features in 5 correlated groups, only
3 of them carrying signal, and just 40 training rows.

In [ ]:
from dolcestat.metrics import LinearRegressionAnalyzer

rng = np.random.default_rng(7)
n_train, n_test, p = 40, 400, 25

# 5 latent factors, each spawning 5 near-duplicate observed features
factors = rng.normal(0, 1, (n_train + n_test, 5))
Z = np.column_stack([factors[:, g] + 0.35 * rng.normal(0, 1, n_train + n_test)
                     for g in range(5) for _ in range(5)])
beta = np.zeros(p)
beta[[0, 5, 10]] = [3.0, -2.0, 1.5]          # only x0, x5, x10 carry signal
target = Z @ beta + 2.0 + rng.normal(0, 1.5, n_train + n_test)

wide_df = pl.DataFrame({f"x{j}": Z[:, j] for j in range(p)} | {"y": target})

train = DolceSet()
train.load_from_polars_dataframe(wide_df.head(n_train), target_col="y")
train.scale("standardize")                    # penalties are only fair on a common scale
stats = train.scaling_info["details"][0]["stats"]

test = DolceSet()
test.load_from_polars_dataframe(wide_df.tail(n_test), target_col="y")
test.scale("standardize", mean=stats["mean"], std=stats["std"])   # reuse train stats

print(f"{n_train} training rows, {p} features")
print(f"correlation within a group  x0~x1  = {np.corrcoef(Z[:, 0], Z[:, 1])[0, 1]:.2f}")
print(f"correlation across groups   x0~x5  = {np.corrcoef(Z[:, 0], Z[:, 5])[0, 1]:.2f}")

## Four fits: none, L1, L2, ElasticNet

Same optimizer, same learning rate, same iteration budget — only the penalty
changes. Watch the gap between training and test error.

In [ ]:
penalties = {
    "none":       {},
    "L1 (Lasso)": {"l1": 0.30},
    "L2 (Ridge)": {"l2": 0.60},
    "ElasticNet": {"l1": 0.15, "l2": 0.30},
}

fits = {}
for name, kwargs in penalties.items():
    gd = GradientDescent(data=train, loss_function="mse", flavor="batch",
                         alpha=0.05, n_iters=20000, tol=1e-10, **kwargs)
    gd.fit()
    fits[name] = gd

header = f"{'penalty':<12} {'train MSE':>10} {'test MSE':>10} {'test R²':>9} {'Σ|w|':>8}"
print(header)
print("-" * len(header))
for name, gd in fits.items():
    train_metrics = LinearRegressionAnalyzer(train.y, gd.predict(train), p)
    test_metrics = LinearRegressionAnalyzer(test.y, gd.predict(test), p)
    weights = gd.get_weights(iteration=-1)
    print(f"{name:<12} {train_metrics.mse():>10.3f} {test_metrics.mse():>10.3f} "
          f"{test_metrics.r2():>9.3f} {np.abs(weights[:-1]).sum():>8.2f}")

The unpenalized fit wins on training data and loses badly on test data —
textbook overfitting. Every penalty gives up training accuracy and is repaid
several times over on unseen rows.

The coefficient profiles show *how* each penalty buys that. Note also that the
bias is identical across all four fits: it is excluded from the penalty, so it
stays at the training mean of y no matter how hard the weights are shrunk.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True, sharey=True)
signal = [0, 5, 10]
for ax, (name, gd) in zip(axes, fits.items()):
    weights = gd.get_weights(iteration=-1)[:-1]
    colors = ["tab:red" if j in signal else "tab:blue" for j in range(p)]
    ax.bar(range(p), weights, color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel(name, rotation=0, ha="right", va="center")
axes[-1].set_xticks(range(p))
axes[-1].set_xticklabels([f"x{j}" for j in range(p)], rotation=90, fontsize=7)
axes[0].set_title("Coefficients per penalty (red = the 3 features that carry signal)")
fig.tight_layout()

print(f"{'penalty':<12} {'bias':>6} | of the 22 noise features: {'|w| < 0.05':>11} {'exactly 0':>11}")
for name, gd in fits.items():
    weights = gd.get_weights(iteration=-1)
    noise = np.delete(weights[:-1], signal)
    print(f"{name:<12} {weights[-1]:>6.2f} | {'':<24} {int((np.abs(noise) < 0.05).sum()):>7}/22 "
          f"{int((noise == 0).sum()):>10}/22")
print(f"\ntraining mean of y = {train.y.mean():.2f}")

Read the three behaviours off the plot:

- **L1** flattens most noise coefficients and keeps a few large ones — it picks
  one representative per correlated group rather than splitting weight among
  near-duplicates.
- **L2** shrinks everything smoothly and *shares* weight across correlated
  features instead of choosing between them.
- **ElasticNet** sits in between, which is usually the safest default when
  features come in correlated groups.

One honest caveat: the `|w| < 0.05` column is large for L1 but the `exactly 0`
column is **zero**. `dolcestat` applies L1 as a subgradient term added to the
gradient, so weights are pushed into a tight band around zero and then oscillate
there — they never land on it. That is enough to suppress a feature's influence,
but it is not the exact sparsity a proximal (soft-thresholding) solver gives you,
so don't use `w == 0` as a feature-selection test here.

A note on reading the loss curve: the tracked loss **includes** the penalty, so
it is the function each run is genuinely descending on and `OptimizerAnalyzer`
is a valid convergence diagnostic for any `l1`/`l2` setting. What it is *not* is
a model-quality comparison across settings — each penalty defines a different
objective, so their curve heights aren't on a common scale. Judge generalization
with held-out error, as in the table above. Expect the L1 curve to stop
decreasing monotonically near the optimum, too: subgradient steps oscillate in a
band of width ~`alpha * l1` rather than settling, which is the same mechanism
that keeps the coefficients off exactly zero.

## Recap

Optimizer choice matters most when the problem is ill-conditioned: momentum buys
you roughly √κ over κ iterations, and Newton's use of the Hessian makes it
condition-invariant — while a quick standardize often removes the problem
altogether. When the difficulty is overfitting rather than conditioning, `l1`,
`l2` and their ElasticNet combination trade training fit for held-out accuracy,
on any flavor and with any momentum setting. These are the very engines the model
wrappers run for you (pass a configured one via `optimizer=`; see notebooks
02–03).

Next, a model with no optimizer at all: [`05_knn`](05_knn.ipynb).